In [1]:
import os, io, base64, warnings
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, Dict, Any, List, Union, Tuple

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, log_loss, brier_score_loss, matthews_corrcoef,
    confusion_matrix, classification_report, roc_curve, precision_recall_curve,
    hamming_loss, jaccard_score, zero_one_loss
)
from sklearn.calibration import calibration_curve
from sklearn.inspection import PartialDependenceDisplay
from sklearn.metrics import det_curve

import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt

import shap

# Counterfactuals
import dice_ml
from dice_ml import Dice


# -------------------------
# Helpers
# -------------------------
def _to_df(X, feature_names=None) -> pd.DataFrame:
    if isinstance(X, pd.DataFrame):
        return X.copy()
    X = np.asarray(X)
    if feature_names is None:
        feature_names = [f"f{i}" for i in range(X.shape[1])]
    return pd.DataFrame(X, columns=feature_names)

def safe_mape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = np.maximum(np.abs(y_true), eps)
    return np.mean(np.abs((y_true - y_pred) / denom)) * 100.0

def smape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = np.maximum((np.abs(y_true) + np.abs(y_pred)), eps)
    return np.mean(2.0 * np.abs(y_pred - y_true) / denom) * 100.0

def psi(expected: pd.Series, actual: pd.Series, buckets=10, eps=1e-6) -> float:
    expected = expected.dropna()
    actual = actual.dropna()

    quantiles = np.quantile(expected, np.linspace(0, 1, buckets + 1))
    quantiles = np.unique(quantiles)
    if len(quantiles) < 3:
        return np.nan

    exp_counts, _ = np.histogram(expected, bins=quantiles)
    act_counts, _ = np.histogram(actual, bins=quantiles)

    exp_perc = exp_counts / max(exp_counts.sum(), 1)
    act_perc = act_counts / max(act_counts.sum(), 1)

    exp_perc = np.clip(exp_perc, eps, 1)
    act_perc = np.clip(act_perc, eps, 1)

    return float(np.sum((act_perc - exp_perc) * np.log(act_perc / exp_perc)))

def _bin_table_binary(y_true, y_prob, X_raw=None, bins=10) -> pd.DataFrame:
    y_true = np.asarray(y_true).reshape(-1)
    y_prob = np.asarray(y_prob).reshape(-1)

    if X_raw is not None:
        X_raw = _to_df(X_raw)
        if len(X_raw) != len(y_true):
            raise ValueError("X_raw rows must match y_true length.")

    df = pd.DataFrame({"pred_prob": y_prob, "y": y_true})
    if X_raw is not None:
        df = pd.concat([df, X_raw.reset_index(drop=True)], axis=1)

    df = df.sort_values("pred_prob", ascending=False).reset_index(drop=True)
    splits = np.array_split(df, bins)

    total_resp = df["y"].sum()
    rand_mean = df["y"].mean()

    raw_cols = list(X_raw.columns) if X_raw is not None else []
    rows = []

    cum_resp = 0
    cum_n = 0

    for i, chunk in enumerate(splits, start=1):
        n = len(chunk)
        if n == 0:
            continue

        responders = chunk["y"].sum()
        cum_resp += responders
        cum_n += n

        pred_mean = chunk["pred_prob"].mean()
        actual_mean = chunk["y"].mean()

        cum_precision = cum_resp / max(cum_n, 1)
        cum_recall = cum_resp / max(total_resp, 1)
        lift = cum_precision / max(rand_mean, 1e-12)

        row = {
            "BIN": i,
            "N": int(n),
            "PRED_MEAN": round(float(pred_mean), 6),
            "ACTUAL_MEAN": round(float(actual_mean), 6),
            "RESPONDERS": int(responders),
            "CUM_RESPONDERS": int(cum_resp),
            "CUM_PRECISION": round(float(cum_precision), 6),
            "CUM_RECALL": round(float(cum_recall), 6),
            "LIFT": round(float(lift), 6),
            "PRED_MIN": round(float(chunk["pred_prob"].min()), 6),
            "PRED_MAX": round(float(chunk["pred_prob"].max()), 6),
        }

        for c in raw_cols:
            row[f"{c} MEAN"] = round(float(chunk[c].mean()), 6)

        rows.append(row)

    out = pd.DataFrame(rows)
    return out

def ks_table_binary(y_true, y_prob, n_bins=10) -> pd.DataFrame:
    df = pd.DataFrame({"y": y_true, "p": y_prob}).dropna()
    df = df.sort_values("p", ascending=False).reset_index(drop=True)
    df["decile"] = pd.qcut(df.index + 1, q=n_bins, labels=False) + 1

    out = []
    total_pos = (df["y"] == 1).sum()
    total_neg = (df["y"] == 0).sum()
    cum_pos = 0
    cum_neg = 0

    for d in range(1, n_bins + 1):
        sub = df[df["decile"] == d]
        pos = (sub["y"] == 1).sum()
        neg = (sub["y"] == 0).sum()
        cum_pos += pos
        cum_neg += neg

        tpr = cum_pos / max(total_pos, 1)
        fpr = cum_neg / max(total_neg, 1)
        ks = abs(tpr - fpr)

        out.append({
            "DECILE": int(d),
            "N": int(len(sub)),
            "POS": int(pos),
            "NEG": int(neg),
            "CUM_POS": int(cum_pos),
            "CUM_NEG": int(cum_neg),
            "TPR": float(tpr),
            "FPR": float(fpr),
            "KS": float(ks),
            "P_MIN": float(sub["p"].min()),
            "P_MAX": float(sub["p"].max()),
            "P_MEAN": float(sub["p"].mean()),
        })

    return pd.DataFrame(out)

def _maybe_import_phik():
    try:
        import phik  # noqa
        return True
    except Exception:
        return False

def _df_to_html_table(df: pd.DataFrame, max_rows=500) -> str:
    dfx = df.copy()
    if len(dfx) > max_rows:
        dfx = dfx.head(max_rows)
    return dfx.to_html(index=False, escape=False)

def _plotly_to_div(fig: go.Figure) -> str:
    return fig.to_html(full_html=False, include_plotlyjs="cdn")

def _mpl_fig_to_img_tag(fig: plt.Figure) -> str:
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=160)
    plt.close(fig)
    b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    return f"<img src='data:image/png;base64,{b64}' style='max-width:100%;height:auto;'/>"

def _wrap_html(title: str, sections: List[Tuple[str, str]]) -> str:
    # sections: [(heading, html_body), ...]
    sec_html = ""
    for h, body in sections:
        sec_html += f"<h2>{h}</h2>\n<div class='section'>{body}</div>\n"
    return f"""
<!doctype html>
<html>
<head>
  <meta charset="utf-8"/>
  <title>{title}</title>
  <style>
    body {{ font-family: Arial, sans-serif; margin: 24px; }}
    h1 {{ margin-bottom: 8px; }}
    h2 {{ margin-top: 22px; }}
    .meta {{ color: #555; margin-bottom: 18px; }}
    .section {{ margin: 14px 0; }}
    .code {{ background: #f6f6f6; padding: 10px; border-radius: 8px; overflow-x: auto; }}
    table {{ border-collapse: collapse; width: 100%; }}
    th, td {{ border: 1px solid #ddd; padding: 8px; font-size: 12px; }}
    th {{ background: #f2f2f2; }}
  </style>
</head>
<body>
  <h1>{title}</h1>
  {sec_html}
</body>
</html>
""".strip()

def _save_html(path: Union[str, Path], html: str) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(html, encoding="utf-8")
    return path


# -------------------------
# Report generator (5 files)
# -------------------------
@dataclass
class FidxReportConfig:
    problem_type: str  # "classification" or "regression"
    bins: int = 10                 # 10=decile, 100=percentile, or any N
    shap_sample: int = 2000
    threshold_grid: int = 19       # number of thresholds in sweep


class FidxFiveReports:
    """
    Generates exactly 5 HTML files:
      #1 Standard Metrics
      #2 Deep Evaluation / Diagnostics
      #3 Interpretability (Global + Local)
      #4 Counterfactual Explanation
      #5 Drift & Data Quality
    """
    def __init__(self, cfg: FidxReportConfig):
        self.cfg = cfg
        self.model = None
        self.feature_names: Optional[List[str]] = None
        self.train_df: Optional[pd.DataFrame] = None
        self.target_name: str = "target"

    def fit(
        self,
        model,
        X_train: Union[pd.DataFrame, np.ndarray],
        y_train: Union[pd.Series, np.ndarray],
        train_df: Optional[pd.DataFrame] = None,
        target_name: str = "target"
    ):
        self.model = model
        self.target_name = target_name
        Xtr = _to_df(X_train)
        self.feature_names = list(Xtr.columns)

        if train_df is not None:
            self.train_df = train_df.copy()
        else:
            self.train_df = Xtr.copy()
            self.train_df[target_name] = np.asarray(y_train).reshape(-1)
        return self

    # --------- User entry point ----------
    def generate(
        self,
        output_dir: Union[str, Path],
        run_name: str,
        X_test: Union[pd.DataFrame, np.ndarray],
        y_test: Union[pd.Series, np.ndarray],
        X_train: Optional[Union[pd.DataFrame, np.ndarray]] = None,
        y_train: Optional[Union[pd.Series, np.ndarray]] = None,
        X_train_raw: Optional[Union[pd.DataFrame, np.ndarray]] = None,
        X_test_raw: Optional[Union[pd.DataFrame, np.ndarray]] = None,
        which_reports: Union[str, List[int]] = "all",  # "all" or [1,3,5]
        report_on: str = "test",                       # "train" | "test" | "both" (applies to #2 bins)
        shap_local_index: int = 0,
        pdp_features: Optional[List[Union[int, str]]] = None,
        drift_ref: Optional[Union[pd.DataFrame, np.ndarray]] = None,
        drift_cur: Optional[Union[pd.DataFrame, np.ndarray]] = None,
        show_progress: bool = True,
    ) -> Dict[int, Path]:

        if which_reports == "all":
            selected = [1, 2, 3, 4, 5]
        else:
            selected = sorted(set(which_reports))

        out_dir = Path(output_dir) / run_name
        out_dir.mkdir(parents=True, exist_ok=True)

        Xte = _to_df(X_test, feature_names=self.feature_names)
        yte = np.asarray(y_test).reshape(-1)

        # precompute preds
        if self.cfg.problem_type == "regression":
            y_pred = np.asarray(self.model.predict(Xte)).reshape(-1)
            y_proba = None
            y_hat = y_pred
            is_binary = False
        else:
            y_hat = self.model.predict(Xte)
            y_pred = y_hat
            y_proba = self.model.predict_proba(Xte) if hasattr(self.model, "predict_proba") else None
            is_binary = (len(np.unique(yte)) == 2) and (y_proba is not None)

        tasks = [(1, "Report #1 – Standard Metrics"),
                 (2, "Report #2 – Deep Evaluation / Diagnostics"),
                 (3, "Report #3 – Interpretability (Global + Local)"),
                 (4, "Report #4 – Counterfactual Explanations"),
                 (5, "Report #5 – Drift & Data Quality")]

        tasks = [t for t in tasks if t[0] in selected]
        iterator = tqdm(tasks, desc="Generating 5 reports") if show_progress else tasks

        saved: Dict[int, Path] = {}

        for rep_id, rep_name in iterator:
            if show_progress and hasattr(iterator, "set_postfix_str"):
                iterator.set_postfix_str(rep_name)

            if rep_id == 1:
                path = self._report1_standard_metrics(out_dir, yte, y_pred, y_proba, is_binary)
                saved[1] = path

            elif rep_id == 2:
                path = self._report2_deep_diagnostics(
                    out_dir, Xte, yte, y_pred, y_proba, is_binary,
                    X_train=X_train, y_train=y_train,
                    X_train_raw=X_train_raw, X_test_raw=X_test_raw,
                    report_on=report_on
                )
                saved[2] = path

            elif rep_id == 3:
                path = self._report3_interpretability(
                    out_dir, Xte, yte, y_proba, is_binary,
                    shap_local_index=shap_local_index,
                    pdp_features=pdp_features
                )
                saved[3] = path

            elif rep_id == 4:
                path = self._report4_counterfactuals(out_dir, Xte, yte, is_binary, shap_local_index)
                saved[4] = path

            elif rep_id == 5:
                path = self._report5_drift_quality(out_dir, Xte, yte, drift_ref, drift_cur)
                saved[5] = path

        return saved

    # ---------- REPORT #1 ----------
    def _report1_standard_metrics(self, out_dir: Path, y_true, y_pred, y_proba, is_binary) -> Path:
        sections = []

        if self.cfg.problem_type == "regression":
            metrics = {
                "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
                "MSE": float(mean_squared_error(y_true, y_pred)),
                "MAE": float(mean_absolute_error(y_true, y_pred)),
                "MAPE(%)": float(safe_mape(y_true, y_pred)),
                "SMAPE(%)": float(smape(y_true, y_pred)),
                "R2": float(r2_score(y_true, y_pred)),
            }
            txt = "\n".join([f"{k}: {v:.6f}" for k, v in metrics.items()])
            sections.append(("Core Regression Metrics", f"<pre class='code'>{txt}</pre>"))

        else:
            # Standard classification metrics (your exact list where applicable)
            core = {
                "Accuracy": float(accuracy_score(y_true, y_pred)),
                "MCC": float(matthews_corrcoef(y_true, y_pred)),
            }

            # these require probability (binary)
            if is_binary:
                p = y_proba[:, 1]
                core.update({
                    "Precision": float(precision_score(y_true, y_pred, zero_division=0)),
                    "Recall": float(recall_score(y_true, y_pred, zero_division=0)),
                    "F1": float(f1_score(y_true, y_pred, zero_division=0)),
                    "AU-ROC": float(roc_auc_score(y_true, p)),
                    "Log Loss": float(log_loss(y_true, y_proba)),
                    "Brier Score": float(brier_score_loss(y_true, p)),
                })

                # losses you listed
                core.update({
                    "Hamming Loss": float(hamming_loss(y_true, y_pred)),
                    "Jaccard Loss": float(1.0 - jaccard_score(y_true, y_pred, average="binary")),
                    "Zero-One Loss": float(zero_one_loss(y_true, y_pred)),
                })

            txt = "\n".join([f"{k}: {v:.6f}" for k, v in core.items()])
            sections.append(("Core Classification Metrics", f"<pre class='code'>{txt}</pre>"))
            rep = classification_report(y_true, y_pred, zero_division=0)
            sections.append(("Classification Report", f"<pre class='code'>{rep}</pre>"))

        html = _wrap_html("Report #1 – Standard Metrics", sections)
        return _save_html(out_dir / "report_1_standard_metrics.html", html)

    # ---------- REPORT #2 ----------
    def _report2_deep_diagnostics(
        self,
        out_dir: Path,
        X_test_df: pd.DataFrame,
        y_test,
        y_pred,
        y_proba,
        is_binary: bool,
        X_train=None,
        y_train=None,
        X_train_raw=None,
        X_test_raw=None,
        report_on="test"
    ) -> Path:

        sections = []
        bins = self.cfg.bins
        bin_label = "Decile" if bins == 10 else "Percentile" if bins == 100 else f"Bin({bins})"

        if self.cfg.problem_type == "regression":
            # decile/percentile distribution by predicted value
            test_bins, _ = _bin_table_regression(y_test, y_pred, X_raw=X_test_raw, bins=bins)
            sections.append((f"{bin_label} Distribution Report (TEST)", _df_to_html_table(test_bins)))

            # residual charts
            fig1 = px.scatter(x=y_pred, y=(np.asarray(y_test) - np.asarray(y_pred)),
                              labels={"x": "Predicted", "y": "Residual"},
                              title="Residual Plot")
            fig1.add_hline(y=0, line_dash="dash")
            sections.append(("Residual Plot", _plotly_to_div(fig1)))

        else:
            if not is_binary:
                sections.append(("Note", "<pre class='code'>Deep diagnostics are implemented for binary classification with predict_proba().</pre>"))
                html = _wrap_html("Report #2 – Deep Evaluation / Diagnostics", sections)
                return _save_html(out_dir / "report_2_deep_diagnostics.html", html)

            p = y_proba[:, 1]

            # 1) Decile/Percentile Distribution Report (Fidx-style)
            test_bins = _bin_table_binary(y_test, p, X_raw=X_test_raw, bins=bins)
            # Rename BIN column label for display
            if "BIN" in test_bins.columns:
                test_bins = test_bins.rename(columns={"BIN": bin_label.upper()})
            sections.append((f"{bin_label} Distribution Report (TEST)", _df_to_html_table(test_bins)))

            # optional train bins
            if report_on in ("train", "both"):
                if X_train is None or y_train is None:
                    raise ValueError("Need X_train,y_train for train diagnostics")
                Xtr = _to_df(X_train, feature_names=self.feature_names)
                ytr = np.asarray(y_train).reshape(-1)
                proba_tr = self.model.predict_proba(Xtr)[:, 1]
                train_bins = _bin_table_binary(ytr, proba_tr, X_raw=X_train_raw, bins=bins)
                if "BIN" in train_bins.columns:
                    train_bins = train_bins.rename(columns={"BIN": bin_label.upper()})
                sections.append((f"{bin_label} Distribution Report (TRAIN)", _df_to_html_table(train_bins)))

            # 2) KS Decile Report (classic uses 10 bins; keep fixed)
            ks_df = ks_table_binary(y_test, p, n_bins=10)
            sections.append(("KS Decile Report", _df_to_html_table(ks_df)))

            # 3) ROC Curve
            fpr, tpr, _ = roc_curve(y_test, p)
            roc_fig = go.Figure()
            roc_fig.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines", name="ROC"))
            roc_fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Random", line=dict(dash="dash")))
            roc_fig.update_layout(title="ROC Curve", xaxis_title="False Positive Rate", yaxis_title="True Positive Rate")
            sections.append(("ROC Curve", _plotly_to_div(roc_fig)))

            # 4) DET Curve
            fpr_det, fnr_det, _ = det_curve(y_test, p)
            det_fig = go.Figure()
            det_fig.add_trace(go.Scatter(x=fpr_det, y=fnr_det, mode="lines", name="DET"))
            det_fig.update_layout(title="DET Curve", xaxis_title="False Positive Rate", yaxis_title="False Negative Rate")
            sections.append(("DET Curve", _plotly_to_div(det_fig)))

            # 5) Cumulative Gain Plot
            df_gain = pd.DataFrame({"y": y_test, "p": p}).sort_values("p", ascending=False).reset_index(drop=True)
            df_gain["cum_pos"] = (df_gain["y"] == 1).cumsum()
            total_pos = (df_gain["y"] == 1).sum()
            df_gain["cum_gain"] = df_gain["cum_pos"] / max(total_pos, 1)
            df_gain["population"] = (df_gain.index + 1) / len(df_gain)

            gain_fig = go.Figure()
            gain_fig.add_trace(go.Scatter(x=df_gain["population"], y=df_gain["cum_gain"], mode="lines", name="Model"))
            gain_fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Random", line=dict(dash="dash")))
            gain_fig.update_layout(title="Cumulative Gain", xaxis_title="Population fraction", yaxis_title="Cumulative gain")
            sections.append(("Cumulative Gain Plot", _plotly_to_div(gain_fig)))

            # 6) Calibration Chart
            prob_true, prob_pred = calibration_curve(y_test, p, n_bins=10, strategy="quantile")
            cal_fig = go.Figure()
            cal_fig.add_trace(go.Scatter(x=prob_pred, y=prob_true, mode="lines+markers", name="Model"))
            cal_fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Perfect", line=dict(dash="dash")))
            cal_fig.update_layout(title="Calibration Chart", xaxis_title="Predicted probability", yaxis_title="Observed frequency")
            sections.append(("Calibration Chart", _plotly_to_div(cal_fig)))

            # 7) Dynamic Threshold Analysis
            thresholds = np.linspace(0.05, 0.95, self.cfg.threshold_grid)
            rows = []
            for t in thresholds:
                pred_t = (p >= t).astype(int)
                rows.append({
                    "threshold": float(t),
                    "precision": float(precision_score(y_test, pred_t, zero_division=0)),
                    "recall": float(recall_score(y_test, pred_t, zero_division=0)),
                    "f1": float(f1_score(y_test, pred_t, zero_division=0)),
                    "accuracy": float(accuracy_score(y_test, pred_t)),
                    "mcc": float(matthews_corrcoef(y_test, pred_t)),
                })
            thr_df = pd.DataFrame(rows)
            thr_fig = px.line(thr_df, x="threshold", y=["precision", "recall", "f1", "accuracy", "mcc"],
                              title="Dynamic Threshold Analysis")
            sections.append(("Dynamic Threshold Analysis", _plotly_to_div(thr_fig)))

            # 8) Multicollinearity report (VIF + corr heatmap)
            # VIF requires statsmodels; skip if not installed
            vif_html = "<pre class='code'>statsmodels not installed; VIF skipped.</pre>"
            try:
                from statsmodels.stats.outliers_influence import variance_inflation_factor
                Xnum = X_test_df.select_dtypes(include=[np.number]).dropna()
                if Xnum.shape[1] >= 2:
                    vifs = []
                    vals = Xnum.to_numpy()
                    for i, col in enumerate(Xnum.columns):
                        vifs.append({"feature": col, "VIF": float(variance_inflation_factor(vals, i))})
                    vif_df = pd.DataFrame(vifs).sort_values("VIF", ascending=False)
                    vif_html = _df_to_html_table(vif_df)
                else:
                    vif_html = "<pre class='code'>Not enough numeric features for VIF.</pre>"
            except Exception as e:
                vif_html = f"<pre class='code'>VIF skipped: {e}</pre>"
            sections.append(("Multicollinearity – VIF", vif_html))

            corr = X_test_df.select_dtypes(include=[np.number]).corr()
            if corr.shape[0] > 1:
                corr_fig = px.imshow(corr, text_auto=False, title="Correlation Heatmap (Numeric Features)")
                sections.append(("Multicollinearity – Correlation Heatmap", _plotly_to_div(corr_fig)))
            else:
                sections.append(("Multicollinearity – Correlation Heatmap", "<pre class='code'>Not enough numeric features.</pre>"))

            # 9) Phik correlation analysis (optional)
            if _maybe_import_phik():
                try:
                    import phik  # noqa
                    # build dataset with target for phik
                    tmp = X_test_df.copy()
                    tmp["target"] = y_test
                    phik_mat = tmp.phik_matrix()
                    phik_fig = px.imshow(phik_mat, title="Phik Correlation Matrix")
                    sections.append(("Phik Correlation Analysis", _plotly_to_div(phik_fig)))
                except Exception as e:
                    sections.append(("Phik Correlation Analysis", f"<pre class='code'>Phik failed: {e}</pre>"))
            else:
                sections.append(("Phik Correlation Analysis", "<pre class='code'>phik not installed; skipped. (pip install phik)</pre>"))

        html = _wrap_html("Report #2 – Deep Evaluation / Diagnostics", sections)
        return _save_html(out_dir / "report_2_deep_diagnostics.html", html)

    # ---------- REPORT #3 ----------
    def _report3_interpretability(
        self,
        out_dir: Path,
        X_test_df: pd.DataFrame,
        y_test,
        y_proba,
        is_binary: bool,
        shap_local_index: int = 0,
        pdp_features: Optional[List[Union[int, str]]] = None
    ) -> Path:
        sections = []

        # GLOBAL SHAP
        Xs = X_test_df.sample(min(len(X_test_df), self.cfg.shap_sample), random_state=42)

        if self.cfg.problem_type == "classification" and hasattr(self.model, "predict_proba"):
            f = lambda data: self.model.predict_proba(pd.DataFrame(data, columns=self.feature_names))
            explainer = shap.Explainer(f, Xs)
            sv = explainer(Xs)

            plt.figure(figsize=(10, 6))
            if sv.values.ndim == 3 and is_binary:
                shap.summary_plot(sv.values[:, :, 1], Xs, show=False)
            else:
                shap.summary_plot(sv, Xs, show=False)
            sections.append(("Global Interpretability – SHAP Summary", _mpl_fig_to_img_tag(plt.gcf())))
        else:
            explainer = shap.Explainer(self.model.predict, Xs)
            sv = explainer(Xs)

            plt.figure(figsize=(10, 6))
            shap.summary_plot(sv, Xs, show=False)
            sections.append(("Global Interpretability – SHAP Summary", _mpl_fig_to_img_tag(plt.gcf())))

        # LOCAL SHAP
        idx = int(shap_local_index)
        idx = max(0, min(idx, len(X_test_df) - 1))
        row = X_test_df.iloc[[idx]].copy()

        if self.cfg.problem_type == "classification" and hasattr(self.model, "predict_proba"):
            f = lambda data: self.model.predict_proba(pd.DataFrame(data, columns=self.feature_names))
            explainer_l = shap.Explainer(f, row)
            sv_l = explainer_l(row)

            plt.figure(figsize=(10, 5))
            if sv_l.values.ndim == 3 and is_binary:
                exp = shap.Explanation(
                    values=sv_l.values[0, :, 1],
                    base_values=sv_l.base_values[0, 1],
                    data=row.iloc[0].values,
                    feature_names=self.feature_names,
                )
                shap.plots.waterfall(exp, show=False)
            else:
                shap.plots.waterfall(sv_l[0], show=False)
            sections.append((f"Local Interpretability – SHAP Waterfall (row={idx})", _mpl_fig_to_img_tag(plt.gcf())))
        else:
            explainer_l = shap.Explainer(self.model.predict, row)
            sv_l = explainer_l(row)
            plt.figure(figsize=(10, 5))
            shap.plots.waterfall(sv_l[0], show=False)
            sections.append((f"Local Interpretability – SHAP Waterfall (row={idx})", _mpl_fig_to_img_tag(plt.gcf())))

        # PDP (optional)
        if pdp_features is not None:
            fig, ax = plt.subplots(figsize=(10, 5))
            PartialDependenceDisplay.from_estimator(self.model, X_test_df, features=pdp_features, ax=ax)
            ax.set_title("Partial Dependence Plot(s)")
            sections.append(("Global Interpretability – Partial Dependence Plot(s)", _mpl_fig_to_img_tag(fig)))
        else:
            sections.append(("Partial Dependence Plot(s)", "<pre class='code'>Skipped. Provide pdp_features=[...] to generate.</pre>"))

        # ELI5 / LIME placeholders (optional)
        sections.append(("ELI5 / LIME", "<pre class='code'>Optional: add ELI5 and LIME if installed and needed. SHAP + PDP are generated here.</pre>"))

        html = _wrap_html("Report #3 – Interpretability (Global + Local)", sections)
        return _save_html(out_dir / "report_3_interpretability.html", html)

    # ---------- REPORT #4 ----------
    def _report4_counterfactuals(self, out_dir: Path, X_test_df: pd.DataFrame, y_test, is_binary: bool, shap_local_index: int = 0) -> Path:
        sections = []
        if self.train_df is None:
            sections.append(("Error", "<pre class='code'>train_df not available. Provide train_df to fit() or X_train/y_train.</pre>"))
            html = _wrap_html("Report #4 – Counterfactual Explanations", sections)
            return _save_html(out_dir / "report_4_counterfactuals.html", html)

        # choose a single query instance
        idx = int(shap_local_index)
        idx = max(0, min(idx, len(X_test_df) - 1))
        query = X_test_df.iloc[[idx]].copy()

        # dice data
        continuous_features = [c for c in self.train_df.columns
                               if c != self.target_name and pd.api.types.is_numeric_dtype(self.train_df[c])]

        data = dice_ml.Data(
            dataframe=self.train_df,
            continuous_features=continuous_features,
            outcome_name=self.target_name
        )

        if self.cfg.problem_type == "classification":
            if not is_binary:
                sections.append(("Note", "<pre class='code'>Counterfactuals implemented here for binary classification. Multi-class requires per-class setup.</pre>"))
                html = _wrap_html("Report #4 – Counterfactual Explanations", sections)
                return _save_html(out_dir / "report_4_counterfactuals.html", html)

            m = dice_ml.Model(model=self.model, backend="sklearn", model_type="classifier")

            # Random counterfactuals
            dice_r = Dice(data, m, method="random")
            cf_r = dice_r.generate_counterfactuals(query_instances=query, total_CFs=3, desired_class="opposite")
            cf_r_df = cf_r.cf_examples_list[0].final_cfs_df
            sections.append(("Random Sampling Counterfactuals", _df_to_html_table(cf_r_df)))

            # Genetic counterfactuals
            dice_g = Dice(data, m, method="genetic")
            cf_g = dice_g.generate_counterfactuals(query_instances=query, total_CFs=3, desired_class="opposite")
            cf_g_df = cf_g.cf_examples_list[0].final_cfs_df
            sections.append(("Genetic / Evolutionary Counterfactuals", _df_to_html_table(cf_g_df)))

            sections.append(("Notes", "<pre class='code'>Counterfactual-based global feature importance can be computed by aggregating feature deltas across counterfactuals.</pre>"))

        else:
            m = dice_ml.Model(model=self.model, backend="sklearn", model_type="regressor")
            dice_r = Dice(data, m, method="random")

            # choose a narrow desired range around current prediction
            pred0 = float(self.model.predict(query)[0])
            desired_range = [pred0 * 0.9, pred0 * 1.1]

            cf_r = dice_r.generate_counterfactuals(query_instances=query, total_CFs=3, desired_range=desired_range)
            cf_r_df = cf_r.cf_examples_list[0].final_cfs_df
            sections.append(("Random Sampling Counterfactuals (Regression)", _df_to_html_table(cf_r_df)))
            sections.append(("Notes", f"<pre class='code'>Desired range used: {desired_range}</pre>"))

        html = _wrap_html("Report #4 – Counterfactual Explanations", sections)
        return _save_html(out_dir / "report_4_counterfactuals.html", html)

    # ---------- REPORT #5 ----------
    def _report5_drift_quality(
        self,
        out_dir: Path,
        X_test_df: pd.DataFrame,
        y_test,
        drift_ref=None,
        drift_cur=None
    ) -> Path:
        sections = []

        # Data integrity checks (test)
        n_rows = len(X_test_df)
        missing = X_test_df.isna().mean().sort_values(ascending=False)
        missing_df = pd.DataFrame({"feature": missing.index, "missing_pct": missing.values})

        dup_count = int(X_test_df.duplicated().sum())
        numeric_cols = X_test_df.select_dtypes(include=[np.number]).columns.tolist()

        sections.append(("Data Integrity – Summary",
                         f"<pre class='code'>Rows: {n_rows}\nDuplicate rows: {dup_count}\nNumeric features: {len(numeric_cols)}</pre>"))
        sections.append(("Data Integrity – Missingness (Top 50)", _df_to_html_table(missing_df.head(50))))

        # Simple outlier scan for numeric columns (z-score based)
        outlier_rows = []
        Xnum = X_test_df[numeric_cols].dropna()
        if len(numeric_cols) > 0 and len(Xnum) > 20:
            z = (Xnum - Xnum.mean()) / (Xnum.std(ddof=0).replace(0, np.nan))
            outlier_rate = (np.abs(z) > 4).mean().sort_values(ascending=False)
            outlier_df = pd.DataFrame({"feature": outlier_rate.index, "outlier_rate(|z|>4)": outlier_rate.values})
            sections.append(("Data Quality – Outlier Rates (|z|>4)", _df_to_html_table(outlier_df.head(50))))
        else:
            sections.append(("Data Quality – Outliers", "<pre class='code'>Not enough numeric data to compute outliers.</pre>"))

        # Drift via PSI (if provided)
        if drift_ref is not None and drift_cur is not None:
            Xr = _to_df(drift_ref, feature_names=self.feature_names)
            Xc = _to_df(drift_cur, feature_names=self.feature_names)

            rows = []
            for col in self.feature_names:
                if pd.api.types.is_numeric_dtype(Xr[col]):
                    rows.append({"feature": col, "psi": psi(Xr[col], Xc[col], buckets=min(self.cfg.bins, 10))})
            drift_df = pd.DataFrame(rows).dropna().sort_values("psi", ascending=False)

            sections.append(("Data Drift – PSI Table", _df_to_html_table(drift_df.head(200))))

            top = drift_df.head(15)
            if len(top) > 0:
                fig = px.bar(top, x="feature", y="psi", title="Top 15 Features by PSI (Higher = More Drift)")
                sections.append(("Data Drift – PSI Plot", _plotly_to_div(fig)))
        else:
            sections.append(("Drift (PSI)", "<pre class='code'>Skipped. Provide drift_ref and drift_cur (e.g., train vs test / OOT).</pre>"))

        # Concept drift placeholder (needs time + monitoring; we keep as explanation)
        sections.append(("Concept Drift",
                         "<pre class='code'>Concept drift requires monitoring P(Y|X) over time. In this lite version, we provide PSI/data drift + integrity checks.\nIf you integrate DeepChecks later, this section can include concept drift checks, stability index, and train-vs-validation comparisons.</pre>"))

        html = _wrap_html("Report #5 – Drift & Data Quality", sections)
        return _save_html(out_dir / "report_5_drift_quality.html", html)


# Regression bin table helper used in report #2
def _bin_table_regression(y_true, y_pred, X_raw=None, bins=10) -> Tuple[pd.DataFrame, pd.DataFrame]:
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    if X_raw is not None:
        X_raw = _to_df(X_raw)
        if len(X_raw) != len(y_true):
            raise ValueError("X_raw rows must match y_true length.")

    df = pd.DataFrame({"pred": y_pred, "y": y_true})
    if X_raw is not None:
        df = pd.concat([df, X_raw.reset_index(drop=True)], axis=1)

    df = df.sort_values("pred", ascending=False).reset_index(drop=True)
    splits = np.array_split(df, bins)

    raw_cols = list(X_raw.columns) if X_raw is not None else []
    rows = []

    for i, chunk in enumerate(splits, start=1):
        n = len(chunk)
        if n == 0:
            continue
        yt = chunk["y"].to_numpy()
        yp = chunk["pred"].to_numpy()

        row = {
            "BIN": i,
            "N": int(n),
            "PRED_MEAN": round(float(np.mean(yp)), 6),
            "ACTUAL_MEAN": round(float(np.mean(yt)), 6),
            "MAE": round(float(mean_absolute_error(yt, yp)), 6),
            "RMSE": round(float(np.sqrt(mean_squared_error(yt, yp))), 6),
            "MAPE(%)": round(float(safe_mape(yt, yp)), 6),
            "SMAPE(%)": round(float(smape(yt, yp)), 6),
            "PRED_MIN": round(float(np.min(yp)), 6),
            "PRED_MAX": round(float(np.max(yp)), 6),
        }
        for c in raw_cols:
            row[f"{c} MEAN"] = round(float(chunk[c].mean()), 6)
        rows.append(row)

    out = pd.DataFrame(rows)
    return out, df


/Users/ajayparasa/.pyenv/versions/ModelXAI_p310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# ============================================================
# FidxAI-Lite: 4 HTML Reports (Report #1 = Model Performance + Diagnostics + Interactive Threshold Slider)
#   - report_1_model_performance.html   (merged old #1 + #2, includes interactive threshold drag bar for binary)
#   - report_3_interpretability.html
#   - report_4_counterfactuals.html
#   - report_5_drift_quality.html
# ============================================================

import os, io, base64, warnings
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, Dict, Any, List, Union, Tuple

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, log_loss, brier_score_loss, matthews_corrcoef,
    confusion_matrix, classification_report, roc_curve,
    precision_recall_curve, hamming_loss, jaccard_score, zero_one_loss
)
from sklearn.calibration import calibration_curve
from sklearn.inspection import PartialDependenceDisplay
from sklearn.metrics import det_curve

import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt

import shap

# Counterfactuals
import dice_ml
from dice_ml import Dice


# -------------------------
# Core helpers
# -------------------------
def _to_df(X, feature_names=None) -> pd.DataFrame:
    if isinstance(X, pd.DataFrame):
        return X.copy()
    X = np.asarray(X)
    if feature_names is None:
        feature_names = [f"f{i}" for i in range(X.shape[1])]
    return pd.DataFrame(X, columns=feature_names)

def safe_mape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = np.maximum(np.abs(y_true), eps)
    return np.mean(np.abs((y_true - y_pred) / denom)) * 100.0

def smape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = np.maximum((np.abs(y_true) + np.abs(y_pred)), eps)
    return np.mean(2.0 * np.abs(y_pred - y_true) / denom) * 100.0

def psi(expected: pd.Series, actual: pd.Series, buckets=10, eps=1e-6) -> float:
    expected = expected.dropna()
    actual = actual.dropna()

    quantiles = np.quantile(expected, np.linspace(0, 1, buckets + 1))
    quantiles = np.unique(quantiles)
    if len(quantiles) < 3:
        return np.nan

    exp_counts, _ = np.histogram(expected, bins=quantiles)
    act_counts, _ = np.histogram(actual, bins=quantiles)

    exp_perc = exp_counts / max(exp_counts.sum(), 1)
    act_perc = act_counts / max(act_counts.sum(), 1)

    exp_perc = np.clip(exp_perc, eps, 1)
    act_perc = np.clip(act_perc, eps, 1)

    return float(np.sum((act_perc - exp_perc) * np.log(act_perc / exp_perc)))

def _maybe_import_phik() -> bool:
    try:
        import phik  # noqa
        return True
    except Exception:
        return False

def _df_to_html_table(df: pd.DataFrame, max_rows=500) -> str:
    dfx = df.copy()
    if len(dfx) > max_rows:
        dfx = dfx.head(max_rows)
    return dfx.to_html(index=False, escape=False)

def _plotly_to_div(fig: go.Figure) -> str:
    return fig.to_html(full_html=False, include_plotlyjs="cdn")

def _mpl_fig_to_img_tag(fig: plt.Figure) -> str:
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=160)
    plt.close(fig)
    b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    return f"<img src='data:image/png;base64,{b64}' style='max-width:100%;height:auto;'/>"

def _wrap_html(title: str, sections: List[Tuple[str, str]]) -> str:
    sec_html = ""
    for h, body in sections:
        sec_html += f"<h2>{h}</h2>\n<div class='section'>{body}</div>\n"
    return f"""
<!doctype html>
<html>
<head>
  <meta charset="utf-8"/>
  <title>{title}</title>
  <style>
    body {{ font-family: Arial, sans-serif; margin: 24px; }}
    h1 {{ margin-bottom: 8px; }}
    h2 {{ margin-top: 22px; }}
    .section {{ margin: 14px 0; }}
    .code {{ background: #f6f6f6; padding: 10px; border-radius: 8px; overflow-x: auto; }}
    table {{ border-collapse: collapse; width: 100%; }}
    th, td {{ border: 1px solid #ddd; padding: 8px; font-size: 12px; }}
    th {{ background: #f2f2f2; }}
  </style>
</head>
<body>
  <h1>{title}</h1>
  {sec_html}
</body>
</html>
""".strip()

def _save_html(path: Union[str, Path], html: str) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(html, encoding="utf-8")
    return path


# -------------------------
# Fidx-style Bin / Decile / Percentile Reports
# -------------------------
def _bin_table_binary(y_true, y_prob, X_raw=None, bins=10) -> pd.DataFrame:
    y_true = np.asarray(y_true).reshape(-1)
    y_prob = np.asarray(y_prob).reshape(-1)

    if X_raw is not None:
        X_raw = _to_df(X_raw)
        if len(X_raw) != len(y_true):
            raise ValueError("X_raw rows must match y_true length.")

    df = pd.DataFrame({"pred_prob": y_prob, "y": y_true})
    if X_raw is not None:
        df = pd.concat([df, X_raw.reset_index(drop=True)], axis=1)

    df = df.sort_values("pred_prob", ascending=False).reset_index(drop=True)
    splits = np.array_split(df, bins)

    total_resp = df["y"].sum()
    rand_mean = df["y"].mean()

    raw_cols = list(X_raw.columns) if X_raw is not None else []
    rows = []

    cum_resp = 0
    cum_n = 0

    for i, chunk in enumerate(splits, start=1):
        n = len(chunk)
        if n == 0:
            continue

        responders = chunk["y"].sum()
        cum_resp += responders
        cum_n += n

        pred_mean = chunk["pred_prob"].mean()
        actual_mean = chunk["y"].mean()

        cum_precision = cum_resp / max(cum_n, 1)
        cum_recall = cum_resp / max(total_resp, 1)
        lift = cum_precision / max(rand_mean, 1e-12)

        row = {
            "BIN": i,
            "N": int(n),
            "PRED_MEAN": round(float(pred_mean), 6),
            "ACTUAL_MEAN": round(float(actual_mean), 6),
            "RESPONDERS": int(responders),
            "CUM_RESPONDERS": int(cum_resp),
            "CUM_PRECISION": round(float(cum_precision), 6),
            "CUM_RECALL": round(float(cum_recall), 6),
            "LIFT": round(float(lift), 6),
            "PRED_MIN": round(float(chunk["pred_prob"].min()), 6),
            "PRED_MAX": round(float(chunk["pred_prob"].max()), 6),
        }

        for c in raw_cols:
            row[f"{c} MEAN"] = round(float(chunk[c].mean()), 6)

        rows.append(row)

    return pd.DataFrame(rows)

def _bin_table_regression(y_true, y_pred, X_raw=None, bins=10) -> pd.DataFrame:
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    if X_raw is not None:
        X_raw = _to_df(X_raw)
        if len(X_raw) != len(y_true):
            raise ValueError("X_raw rows must match y_true length.")

    df = pd.DataFrame({"pred": y_pred, "y": y_true})
    if X_raw is not None:
        df = pd.concat([df, X_raw.reset_index(drop=True)], axis=1)

    df = df.sort_values("pred", ascending=False).reset_index(drop=True)
    splits = np.array_split(df, bins)

    raw_cols = list(X_raw.columns) if X_raw is not None else []
    rows = []

    for i, chunk in enumerate(splits, start=1):
        n = len(chunk)
        if n == 0:
            continue
        yt = chunk["y"].to_numpy()
        yp = chunk["pred"].to_numpy()

        row = {
            "BIN": i,
            "N": int(n),
            "PRED_MEAN": round(float(np.mean(yp)), 6),
            "ACTUAL_MEAN": round(float(np.mean(yt)), 6),
            "MAE": round(float(mean_absolute_error(yt, yp)), 6),
            "RMSE": round(float(np.sqrt(mean_squared_error(yt, yp))), 6),
            "MAPE(%)": round(float(safe_mape(yt, yp)), 6),
            "SMAPE(%)": round(float(smape(yt, yp)), 6),
            "PRED_MIN": round(float(np.min(yp)), 6),
            "PRED_MAX": round(float(np.max(yp)), 6),
        }
        for c in raw_cols:
            row[f"{c} MEAN"] = round(float(chunk[c].mean()), 6)
        rows.append(row)

    return pd.DataFrame(rows)

def ks_table_binary(y_true, y_prob, n_bins=10) -> pd.DataFrame:
    df = pd.DataFrame({"y": y_true, "p": y_prob}).dropna()
    df = df.sort_values("p", ascending=False).reset_index(drop=True)
    df["decile"] = pd.qcut(df.index + 1, q=n_bins, labels=False) + 1

    out = []
    total_pos = (df["y"] == 1).sum()
    total_neg = (df["y"] == 0).sum()
    cum_pos = 0
    cum_neg = 0

    for d in range(1, n_bins + 1):
        sub = df[df["decile"] == d]
        pos = (sub["y"] == 1).sum()
        neg = (sub["y"] == 0).sum()
        cum_pos += pos
        cum_neg += neg

        tpr = cum_pos / max(total_pos, 1)
        fpr = cum_neg / max(total_neg, 1)
        ks = abs(tpr - fpr)

        out.append({
            "DECILE": int(d),
            "N": int(len(sub)),
            "POS": int(pos),
            "NEG": int(neg),
            "CUM_POS": int(cum_pos),
            "CUM_NEG": int(cum_neg),
            "TPR": float(tpr),
            "FPR": float(fpr),
            "KS": float(ks),
            "P_MIN": float(sub["p"].min()),
            "P_MAX": float(sub["p"].max()),
            "P_MEAN": float(sub["p"].mean()),
        })
    return pd.DataFrame(out)


# -------------------------
# Interactive Threshold Slider block (Binary classification only)
# -------------------------
def build_threshold_slider_block(y_true, p, thresholds=None) -> str:
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)

    if thresholds is None:
        thresholds = np.round(np.linspace(0.01, 0.99, 99), 3)

    rows = []
    cms = []
    for t in thresholds:
        y_hat = (p >= t).astype(int)
        cm = confusion_matrix(y_true, y_hat, labels=[0, 1])
        cms.append(cm.tolist())

        acc = accuracy_score(y_true, y_hat)
        prec = precision_score(y_true, y_hat, zero_division=0)
        rec = recall_score(y_true, y_hat, zero_division=0)
        f1v = f1_score(y_true, y_hat, zero_division=0)
        mcc = matthews_corrcoef(y_true, y_hat)
        ham = hamming_loss(y_true, y_hat)
        jac_loss = 1.0 - jaccard_score(y_true, y_hat, average="binary")
        zo = zero_one_loss(y_true, y_hat)

        rows.append([float(t), float(acc), float(prec), float(rec), float(f1v),
                     float(mcc), float(ham), float(jac_loss), float(zo)])

    df = pd.DataFrame(rows, columns=[
        "threshold","accuracy","precision","recall","f1","mcc",
        "hamming_loss","jaccard_loss","zero_one_loss"
    ])

    fig = go.Figure()
    for col in ["accuracy","precision","recall","f1","mcc"]:
        fig.add_trace(go.Scatter(x=df["threshold"], y=df[col], mode="lines", name=col))

    # marker trace (we'll move this)
    fig.add_trace(go.Scatter(
        x=[df["threshold"].iloc[0]],
        y=[df["f1"].iloc[0]],
        mode="markers",
        name="current (F1)",
        marker=dict(size=10)
    ))

    fig.update_layout(
        title="Interactive Dynamic Threshold Analysis (drag slider)",
        xaxis_title="Threshold",
        yaxis_title="Metric value",
        height=450
    )

    # confusion matrix initial heatmap (Plotly HTML)
    cm0 = np.array(cms[0])
    cm_fig = px.imshow(cm0, text_auto=True, title="Confusion Matrix @ threshold",
                       labels=dict(x="Predicted", y="Actual"))
    cm_div = cm_fig.to_html(full_html=False, include_plotlyjs="cdn")

    table_html = """
    <table id="metricTable">
      <thead>
        <tr>
          <th>threshold</th><th>accuracy</th><th>precision</th><th>recall</th><th>f1</th><th>mcc</th>
          <th>hamming_loss</th><th>jaccard_loss</th><th>zero_one_loss</th>
        </tr>
      </thead>
      <tbody>
        <tr>
          <td id="t"></td><td id="acc"></td><td id="prec"></td><td id="rec"></td><td id="f1"></td><td id="mcc"></td>
          <td id="ham"></td><td id="jac"></td><td id="zo"></td>
        </tr>
      </tbody>
    </table>
    """

    div = fig.to_html(full_html=False, include_plotlyjs="cdn", div_id="thrFig")

    thresholds_list = df["threshold"].tolist()
    data_rows = df.values.tolist()

    block = f"""
    <div class="section">
      {div}
      <div style="margin-top:10px;">
        <label><b>Threshold:</b></label>
        <input id="thrSlider" type="range" min="0" max="{len(thresholds_list)-1}" value="0" step="1" style="width:100%;">
        <div style="margin-top:8px;">{table_html}</div>
      </div>

      <div style="margin-top:18px;" id="cmBlock">
        {cm_div}
      </div>
    </div>

    <script>
      const rows = {data_rows};
      const cms = {cms};

      function fmt(x) {{
        return (typeof x === "number") ? x.toFixed(6) : x;
      }}

      function updateTable(i) {{
        const r = rows[i];
        document.getElementById("t").innerText = fmt(r[0]);
        document.getElementById("acc").innerText = fmt(r[1]);
        document.getElementById("prec").innerText = fmt(r[2]);
        document.getElementById("rec").innerText = fmt(r[3]);
        document.getElementById("f1").innerText = fmt(r[4]);
        document.getElementById("mcc").innerText = fmt(r[5]);
        document.getElementById("ham").innerText = fmt(r[6]);
        document.getElementById("jac").innerText = fmt(r[7]);
        document.getElementById("zo").innerText = fmt(r[8]);
      }}

      function updateMarker(i) {{
        const thr = rows[i][0];
        const f1 = rows[i][4];
        Plotly.restyle("thrFig", {{
          x: [[thr]],
          y: [[f1]]
        }}, [5]); // marker trace index = 5
      }}

      function updateCM(i) {{
        const cm = cms[i];
        const gd = document.querySelector("#cmBlock .js-plotly-plot");
        const data = [{{
          z: cm,
          type: "heatmap",
          showscale: true
        }}];
        const layout = {{
          title: "Confusion Matrix @ threshold = " + rows[i][0].toFixed(3),
          xaxis: {{title: "Predicted"}},
          yaxis: {{title: "Actual"}}
        }};
        Plotly.react(gd, data, layout);
      }}

      const slider = document.getElementById("thrSlider");
      slider.addEventListener("input", (e) => {{
        const i = parseInt(e.target.value);
        updateTable(i);
        updateMarker(i);
        updateCM(i);
      }});

      updateTable(0);
    </script>
    """
    return block


# ============================================================
# Main 4-report generator
# ============================================================
@dataclass
class FidxReportConfig:
    problem_type: str  # "classification" or "regression"
    bins: int = 10
    shap_sample: int = 2000


class FidxReports4:
    """
    Writes exactly 4 HTML files:
      Report #1 – Model Performance Evaluation (merged #1+#2 + interactive threshold slider for binary)
      Report #3 – Interpretability (Global + Local)
      Report #4 – Counterfactual Explanations
      Report #5 – Drift & Data Quality
    """
    def __init__(self, cfg: FidxReportConfig):
        self.cfg = cfg
        self.model = None
        self.feature_names: Optional[List[str]] = None
        self.train_df: Optional[pd.DataFrame] = None
        self.target_name: str = "target"

    def fit(
        self,
        model,
        X_train: Union[pd.DataFrame, np.ndarray],
        y_train: Union[pd.Series, np.ndarray],
        train_df: Optional[pd.DataFrame] = None,
        target_name: str = "target"
    ):
        self.model = model
        self.target_name = target_name
        Xtr = _to_df(X_train)
        self.feature_names = list(Xtr.columns)

        if train_df is not None:
            self.train_df = train_df.copy()
        else:
            self.train_df = Xtr.copy()
            self.train_df[target_name] = np.asarray(y_train).reshape(-1)
        return self

    def generate(
        self,
        output_dir: Union[str, Path],
        run_name: str,
        X_test: Union[pd.DataFrame, np.ndarray],
        y_test: Union[pd.Series, np.ndarray],
        X_train: Optional[Union[pd.DataFrame, np.ndarray]] = None,
        y_train: Optional[Union[pd.Series, np.ndarray]] = None,
        X_train_raw: Optional[Union[pd.DataFrame, np.ndarray]] = None,
        X_test_raw: Optional[Union[pd.DataFrame, np.ndarray]] = None,
        which_reports: Union[str, List[int]] = "all",   # "all" or [1,3,5]
        report_on: str = "test",                        # "train" | "test" | "both" (affects bin tables inside Report #1)
        shap_local_index: int = 0,
        pdp_features: Optional[List[Union[int, str]]] = None,
        drift_ref: Optional[Union[pd.DataFrame, np.ndarray]] = None,
        drift_cur: Optional[Union[pd.DataFrame, np.ndarray]] = None,
        show_progress: bool = True,
    ) -> Dict[int, Path]:

        if which_reports == "all":
            selected = [1, 3, 4, 5]
        else:
            selected = sorted(set(which_reports))

        out_dir = Path(output_dir) / run_name
        out_dir.mkdir(parents=True, exist_ok=True)

        Xte = _to_df(X_test, feature_names=self.feature_names)
        yte = np.asarray(y_test).reshape(-1)

        # Predictions
        if self.cfg.problem_type == "regression":
            y_pred = np.asarray(self.model.predict(Xte)).reshape(-1)
            y_proba = None
            is_binary = False
        else:
            y_pred = self.model.predict(Xte)
            y_proba = self.model.predict_proba(Xte) if hasattr(self.model, "predict_proba") else None
            is_binary = (len(np.unique(yte)) == 2) and (y_proba is not None)

        tasks = [(1, "Report #1 – Model Performance Evaluation"),
                 (3, "Report #3 – Interpretability (Global + Local)"),
                 (4, "Report #4 – Counterfactual Explanations"),
                 (5, "Report #5 – Drift & Data Quality")]
        tasks = [t for t in tasks if t[0] in selected]
        iterator = tqdm(tasks, desc="Generating reports") if show_progress else tasks

        saved: Dict[int, Path] = {}

        for rep_id, rep_name in iterator:
            if show_progress and hasattr(iterator, "set_postfix_str"):
                iterator.set_postfix_str(rep_name)

            if rep_id == 1:
                saved[1] = self._report1_model_performance(
                    out_dir, Xte, yte, y_pred, y_proba, is_binary,
                    X_train=X_train, y_train=y_train,
                    X_train_raw=X_train_raw, X_test_raw=X_test_raw,
                    report_on=report_on
                )

            elif rep_id == 3:
                saved[3] = self._report3_interpretability(
                    out_dir, Xte, yte, y_proba, is_binary,
                    shap_local_index=shap_local_index,
                    pdp_features=pdp_features
                )

            elif rep_id == 4:
                saved[4] = self._report4_counterfactuals(
                    out_dir, Xte, yte, is_binary,
                    query_index=shap_local_index
                )

            elif rep_id == 5:
                saved[5] = self._report5_drift_quality(
                    out_dir, Xte, yte,
                    drift_ref=drift_ref,
                    drift_cur=drift_cur
                )

        return saved

    # -------------------------
    # Report #1 (Merged #1 + #2 + Interactive Threshold Slider)
    # -------------------------
    def _report1_model_performance(
        self,
        out_dir: Path,
        X_test_df: pd.DataFrame,
        y_test,
        y_pred,
        y_proba,
        is_binary: bool,
        X_train=None,
        y_train=None,
        X_train_raw=None,
        X_test_raw=None,
        report_on="test"
    ) -> Path:

        sections = []

        bins = self.cfg.bins
        bin_label = "Decile" if bins == 10 else "Percentile" if bins == 100 else f"Bin({bins})"

        if self.cfg.problem_type == "regression":
            # Core regression metrics
            metrics = {
                "RMSE": float(np.sqrt(mean_squared_error(y_test, y_pred))),
                "MSE": float(mean_squared_error(y_test, y_pred)),
                "MAE": float(mean_absolute_error(y_test, y_pred)),
                "MAPE(%)": float(safe_mape(y_test, y_pred)),
                "SMAPE(%)": float(smape(y_test, y_pred)),
                "R2": float(r2_score(y_test, y_pred)),
            }
            txt = "\n".join([f"{k}: {v:.6f}" for k, v in metrics.items()])
            sections.append(("Core Regression Metrics", f"<pre class='code'>{txt}</pre>"))

            # Visuals
            fig_avp = px.scatter(x=y_test, y=y_pred, labels={"x":"Actual","y":"Predicted"}, title="Actual vs Predicted")
            sections.append(("Actual vs Predicted", _plotly_to_div(fig_avp)))

            resid = np.asarray(y_test) - np.asarray(y_pred)
            fig_res = px.scatter(x=y_pred, y=resid, labels={"x":"Predicted","y":"Residual"}, title="Residual Plot")
            fig_res.add_hline(y=0, line_dash="dash")
            sections.append(("Residual Plot", _plotly_to_div(fig_res)))

            # Distribution report
            test_bins = _bin_table_regression(y_test, y_pred, X_raw=X_test_raw, bins=bins).rename(columns={"BIN": bin_label.upper()})
            sections.append((f"{bin_label} Distribution Report (TEST)", _df_to_html_table(test_bins)))

            if report_on in ("train", "both") and X_train is not None and y_train is not None:
                Xtr = _to_df(X_train, feature_names=self.feature_names)
                ytr = np.asarray(y_train).reshape(-1)
                pred_tr = np.asarray(self.model.predict(Xtr)).reshape(-1)
                train_bins = _bin_table_regression(ytr, pred_tr, X_raw=X_train_raw, bins=bins).rename(columns={"BIN": bin_label.upper()})
                sections.append((f"{bin_label} Distribution Report (TRAIN)", _df_to_html_table(train_bins)))

            html = _wrap_html("Report #1 – Model Performance Evaluation (Regression)", sections)
            return _save_html(out_dir / "report_1_model_performance.html", html)

        # -------------------------
        # Classification
        # -------------------------
        rep = classification_report(y_test, y_pred, zero_division=0)
        sections.append(("Classification Report", f"<pre class='code'>{rep}</pre>"))

        core = {
            "Accuracy": float(accuracy_score(y_test, y_pred)),
            "MCC": float(matthews_corrcoef(y_test, y_pred)),
        }

        if is_binary:
            p = y_proba[:, 1]
            core.update({
                "Precision@0.5": float(precision_score(y_test, y_pred, zero_division=0)),
                "Recall@0.5": float(recall_score(y_test, y_pred, zero_division=0)),
                "F1@0.5": float(f1_score(y_test, y_pred, zero_division=0)),
                "AU-ROC": float(roc_auc_score(y_test, p)),
                "Log Loss": float(log_loss(y_test, y_proba)),
                "Brier Score": float(brier_score_loss(y_test, p)),
                "Hamming Loss@0.5": float(hamming_loss(y_test, y_pred)),
                "Jaccard Loss@0.5": float(1.0 - jaccard_score(y_test, y_pred, average="binary")),
                "Zero-One Loss@0.5": float(zero_one_loss(y_test, y_pred)),
            })

        txt = "\n".join([f"{k}: {v:.6f}" for k, v in core.items()])
        sections.append(("Core Classification Metrics", f"<pre class='code'>{txt}</pre>"))

        # Interactive threshold slider
        if is_binary:
            sections.append(("Dynamic Threshold Analysis (Interactive)", build_threshold_slider_block(y_test, y_proba[:, 1])))
        else:
            sections.append(("Dynamic Threshold Analysis (Interactive)", "<pre class='code'>Available for binary classification with predict_proba().</pre>"))

        # Deep diagnostics only for binary
        if is_binary:
            p = y_proba[:, 1]

            # Decile / Percentile distribution (Fidx-style)
            test_bins = _bin_table_binary(y_test, p, X_raw=X_test_raw, bins=bins).rename(columns={"BIN": bin_label.upper()})
            sections.append((f"{bin_label} Distribution Report (TEST)", _df_to_html_table(test_bins)))

            if report_on in ("train", "both"):
                if X_train is None or y_train is None:
                    raise ValueError("Need X_train and y_train when report_on includes 'train' or 'both'.")
                Xtr = _to_df(X_train, feature_names=self.feature_names)
                ytr = np.asarray(y_train).reshape(-1)
                p_tr = self.model.predict_proba(Xtr)[:, 1]
                train_bins = _bin_table_binary(ytr, p_tr, X_raw=X_train_raw, bins=bins).rename(columns={"BIN": bin_label.upper()})
                sections.append((f"{bin_label} Distribution Report (TRAIN)", _df_to_html_table(train_bins)))

            # KS Decile report (fixed 10)
            ks_df = ks_table_binary(y_test, p, n_bins=10)
            sections.append(("KS Decile Report", _df_to_html_table(ks_df)))

            # ROC
            fpr, tpr, _ = roc_curve(y_test, p)
            roc_fig = go.Figure()
            roc_fig.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines", name="ROC"))
            roc_fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode="lines", name="Random", line=dict(dash="dash")))
            roc_fig.update_layout(title="ROC Curve", xaxis_title="FPR", yaxis_title="TPR")
            sections.append(("ROC Curve", _plotly_to_div(roc_fig)))

            # DET
            fpr_det, fnr_det, _ = det_curve(y_test, p)
            det_fig = go.Figure()
            det_fig.add_trace(go.Scatter(x=fpr_det, y=fnr_det, mode="lines", name="DET"))
            det_fig.update_layout(title="DET Curve", xaxis_title="FPR", yaxis_title="FNR")
            sections.append(("DET Curve", _plotly_to_div(det_fig)))

            # Cumulative Gain
            df_gain = pd.DataFrame({"y": y_test, "p": p}).sort_values("p", ascending=False).reset_index(drop=True)
            df_gain["cum_pos"] = (df_gain["y"] == 1).cumsum()
            total_pos = (df_gain["y"] == 1).sum()
            df_gain["cum_gain"] = df_gain["cum_pos"] / max(total_pos, 1)
            df_gain["population"] = (df_gain.index + 1) / len(df_gain)
            gain_fig = go.Figure()
            gain_fig.add_trace(go.Scatter(x=df_gain["population"], y=df_gain["cum_gain"], mode="lines", name="Model"))
            gain_fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode="lines", name="Random", line=dict(dash="dash")))
            gain_fig.update_layout(title="Cumulative Gain", xaxis_title="Population", yaxis_title="Gain")
            sections.append(("Cumulative Gain Plot", _plotly_to_div(gain_fig)))

            # Calibration
            prob_true, prob_pred = calibration_curve(y_test, p, n_bins=10, strategy="quantile")
            cal_fig = go.Figure()
            cal_fig.add_trace(go.Scatter(x=prob_pred, y=prob_true, mode="lines+markers", name="Model"))
            cal_fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode="lines", name="Perfect", line=dict(dash="dash")))
            cal_fig.update_layout(title="Calibration Chart", xaxis_title="Pred prob", yaxis_title="Observed freq")
            sections.append(("Calibration Chart", _plotly_to_div(cal_fig)))

            # Multicollinearity (VIF + corr)
            try:
                from statsmodels.stats.outliers_influence import variance_inflation_factor
                Xnum = X_test_df.select_dtypes(include=[np.number]).dropna()
                if Xnum.shape[1] >= 2:
                    vals = Xnum.to_numpy()
                    vifs = [{"feature": c, "VIF": float(variance_inflation_factor(vals, i))} for i, c in enumerate(Xnum.columns)]
                    vif_df = pd.DataFrame(vifs).sort_values("VIF", ascending=False)
                    sections.append(("Multicollinearity – VIF", _df_to_html_table(vif_df)))
                else:
                    sections.append(("Multicollinearity – VIF", "<pre class='code'>Not enough numeric features for VIF.</pre>"))
            except Exception as e:
                sections.append(("Multicollinearity – VIF", f"<pre class='code'>VIF skipped: {e}</pre>"))

            corr = X_test_df.select_dtypes(include=[np.number]).corr()
            if corr.shape[0] > 1:
                corr_fig = px.imshow(corr, title="Correlation Heatmap (Numeric Features)")
                sections.append(("Multicollinearity – Correlation Heatmap", _plotly_to_div(corr_fig)))
            else:
                sections.append(("Multicollinearity – Correlation Heatmap", "<pre class='code'>Not enough numeric features for correlation.</pre>"))

            # Phik optional
            if _maybe_import_phik():
                try:
                    import phik  # noqa
                    tmp = X_test_df.copy()
                    tmp["target"] = y_test
                    phik_mat = tmp.phik_matrix()
                    phik_fig = px.imshow(phik_mat, title="Phik Correlation Matrix")
                    sections.append(("Phik Correlation Analysis", _plotly_to_div(phik_fig)))
                except Exception as e:
                    sections.append(("Phik Correlation Analysis", f"<pre class='code'>Phik failed: {e}</pre>"))
            else:
                sections.append(("Phik Correlation Analysis", "<pre class='code'>phik not installed; skipped. (pip install phik)</pre>"))

        else:
            sections.append(("Deep Diagnostics", "<pre class='code'>Deep diagnostics (ROC/DET/KS/Calibration/Gain) are built for binary classification with predict_proba().</pre>"))

        html = _wrap_html("Report #1 – Model Performance Evaluation (Classification)", sections)
        return _save_html(out_dir / "report_1_model_performance.html", html)

    # -------------------------
    # Report #3 Interpretability
    # -------------------------
    def _report3_interpretability(
        self,
        out_dir: Path,
        X_test_df: pd.DataFrame,
        y_test,
        y_proba,
        is_binary: bool,
        shap_local_index: int = 0,
        pdp_features: Optional[List[Union[int, str]]] = None
    ) -> Path:
        sections = []

        Xs = X_test_df.sample(min(len(X_test_df), self.cfg.shap_sample), random_state=42)

        # Global SHAP
        if self.cfg.problem_type == "classification" and hasattr(self.model, "predict_proba"):
            f = lambda data: self.model.predict_proba(pd.DataFrame(data, columns=self.feature_names))
            explainer = shap.Explainer(f, Xs)
            sv = explainer(Xs)

            plt.figure(figsize=(10, 6))
            if sv.values.ndim == 3 and is_binary:
                shap.summary_plot(sv.values[:, :, 1], Xs, show=False)
            else:
                shap.summary_plot(sv, Xs, show=False)
            sections.append(("Global Interpretability – SHAP Summary", _mpl_fig_to_img_tag(plt.gcf())))
        else:
            explainer = shap.Explainer(self.model.predict, Xs)
            sv = explainer(Xs)

            plt.figure(figsize=(10, 6))
            shap.summary_plot(sv, Xs, show=False)
            sections.append(("Global Interpretability – SHAP Summary", _mpl_fig_to_img_tag(plt.gcf())))

        # Local SHAP
        idx = int(shap_local_index)
        idx = max(0, min(idx, len(X_test_df) - 1))
        row = X_test_df.iloc[[idx]].copy()

        if self.cfg.problem_type == "classification" and hasattr(self.model, "predict_proba"):
            f = lambda data: self.model.predict_proba(pd.DataFrame(data, columns=self.feature_names))
            explainer_l = shap.Explainer(f, row)
            sv_l = explainer_l(row)

            plt.figure(figsize=(10, 5))
            if sv_l.values.ndim == 3 and is_binary:
                exp = shap.Explanation(
                    values=sv_l.values[0, :, 1],
                    base_values=sv_l.base_values[0, 1],
                    data=row.iloc[0].values,
                    feature_names=self.feature_names,
                )
                shap.plots.waterfall(exp, show=False)
            else:
                shap.plots.waterfall(sv_l[0], show=False)

            sections.append((f"Local Interpretability – SHAP Waterfall (row={idx})", _mpl_fig_to_img_tag(plt.gcf())))
        else:
            explainer_l = shap.Explainer(self.model.predict, row)
            sv_l = explainer_l(row)
            plt.figure(figsize=(10, 5))
            shap.plots.waterfall(sv_l[0], show=False)
            sections.append((f"Local Interpretability – SHAP Waterfall (row={idx})", _mpl_fig_to_img_tag(plt.gcf())))

        # PDP optional
        if pdp_features is not None:
            fig, ax = plt.subplots(figsize=(10, 5))
            PartialDependenceDisplay.from_estimator(self.model, X_test_df, features=pdp_features, ax=ax)
            ax.set_title("Partial Dependence Plot(s)")
            sections.append(("Global Interpretability – Partial Dependence Plot(s)", _mpl_fig_to_img_tag(fig)))
        else:
            sections.append(("Partial Dependence Plot(s)", "<pre class='code'>Skipped. Provide pdp_features=[...] to generate.</pre>"))

        sections.append(("ELI5 / LIME", "<pre class='code'>Optional: add ELI5 and LIME if installed. This report includes SHAP (global+local) + optional PDP.</pre>"))

        html = _wrap_html("Report #3 – Interpretability (Global + Local)", sections)
        return _save_html(out_dir / "report_3_interpretability.html", html)

    # -------------------------
    # Report #4 Counterfactuals (DiCE)
    # -------------------------
    def _report4_counterfactuals(
        self,
        out_dir: Path,
        X_test_df: pd.DataFrame,
        y_test,
        is_binary: bool,
        query_index: int = 0
    ) -> Path:
        sections = []

        if self.train_df is None:
            sections.append(("Error", "<pre class='code'>train_df not available. Provide train_df to fit() or X_train/y_train.</pre>"))
            html = _wrap_html("Report #4 – Counterfactual Explanations", sections)
            return _save_html(out_dir / "report_4_counterfactuals.html", html)

        idx = int(query_index)
        idx = max(0, min(idx, len(X_test_df) - 1))
        query = X_test_df.iloc[[idx]].copy()

        continuous_features = [c for c in self.train_df.columns
                               if c != self.target_name and pd.api.types.is_numeric_dtype(self.train_df[c])]

        data = dice_ml.Data(
            dataframe=self.train_df,
            continuous_features=continuous_features,
            outcome_name=self.target_name
        )

        if self.cfg.problem_type == "classification":
            if not is_binary:
                sections.append(("Note", "<pre class='code'>Counterfactuals implemented here for binary classification. Multi-class needs per-class setup.</pre>"))
                html = _wrap_html("Report #4 – Counterfactual Explanations", sections)
                return _save_html(out_dir / "report_4_counterfactuals.html", html)

            m = dice_ml.Model(model=self.model, backend="sklearn", model_type="classifier")

            # Random CFs
            dice_r = Dice(data, m, method="random")
            cf_r = dice_r.generate_counterfactuals(query_instances=query, total_CFs=3, desired_class="opposite")
            cf_r_df = cf_r.cf_examples_list[0].final_cfs_df
            sections.append(("Random Sampling Counterfactuals", _df_to_html_table(cf_r_df)))

            # Genetic CFs
            dice_g = Dice(data, m, method="genetic")
            cf_g = dice_g.generate_counterfactuals(query_instances=query, total_CFs=3, desired_class="opposite")
            cf_g_df = cf_g.cf_examples_list[0].final_cfs_df
            sections.append(("Genetic / Evolutionary Counterfactuals", _df_to_html_table(cf_g_df)))

            sections.append(("Notes", "<pre class='code'>To get counterfactual-based global importance: aggregate absolute feature deltas across CFs and rank.</pre>"))

        else:
            m = dice_ml.Model(model=self.model, backend="sklearn", model_type="regressor")
            dice_r = Dice(data, m, method="random")

            pred0 = float(self.model.predict(query)[0])
            desired_range = [pred0 * 0.9, pred0 * 1.1]

            cf_r = dice_r.generate_counterfactuals(query_instances=query, total_CFs=3, desired_range=desired_range)
            cf_r_df = cf_r.cf_examples_list[0].final_cfs_df
            sections.append(("Random Sampling Counterfactuals (Regression)", _df_to_html_table(cf_r_df)))
            sections.append(("Notes", f"<pre class='code'>Desired range used: {desired_range}</pre>"))

        html = _wrap_html("Report #4 – Counterfactual Explanations", sections)
        return _save_html(out_dir / "report_4_counterfactuals.html", html)

    # -------------------------
    # Report #5 Drift & Data Quality
    # -------------------------
    def _report5_drift_quality(
        self,
        out_dir: Path,
        X_test_df: pd.DataFrame,
        y_test,
        drift_ref=None,
        drift_cur=None
    ) -> Path:
        sections = []

        n_rows = len(X_test_df)
        missing = X_test_df.isna().mean().sort_values(ascending=False)
        missing_df = pd.DataFrame({"feature": missing.index, "missing_pct": missing.values})

        dup_count = int(X_test_df.duplicated().sum())
        numeric_cols = X_test_df.select_dtypes(include=[np.number]).columns.tolist()

        sections.append(("Data Integrity – Summary",
                         f"<pre class='code'>Rows: {n_rows}\nDuplicate rows: {dup_count}\nNumeric features: {len(numeric_cols)}</pre>"))
        sections.append(("Data Integrity – Missingness (Top 50)", _df_to_html_table(missing_df.head(50))))

        # Outliers (simple z-score)
        Xnum = X_test_df[numeric_cols].dropna()
        if len(numeric_cols) > 0 and len(Xnum) > 20:
            z = (Xnum - Xnum.mean()) / (Xnum.std(ddof=0).replace(0, np.nan))
            outlier_rate = (np.abs(z) > 4).mean().sort_values(ascending=False)
            outlier_df = pd.DataFrame({"feature": outlier_rate.index, "outlier_rate(|z|>4)": outlier_rate.values})
            sections.append(("Data Quality – Outlier Rates (|z|>4)", _df_to_html_table(outlier_df.head(50))))
        else:
            sections.append(("Data Quality – Outliers", "<pre class='code'>Not enough numeric data to compute outliers.</pre>"))

        # Drift via PSI (if provided)
        if drift_ref is not None and drift_cur is not None:
            Xr = _to_df(drift_ref, feature_names=self.feature_names)
            Xc = _to_df(drift_cur, feature_names=self.feature_names)

            rows = []
            for col in self.feature_names:
                if pd.api.types.is_numeric_dtype(Xr[col]):
                    rows.append({"feature": col, "psi": psi(Xr[col], Xc[col], buckets=min(self.cfg.bins, 10))})
            drift_df = pd.DataFrame(rows).dropna().sort_values("psi", ascending=False)

            sections.append(("Data Drift – PSI Table", _df_to_html_table(drift_df.head(200))))

            top = drift_df.head(15)
            if len(top) > 0:
                fig = px.bar(top, x="feature", y="psi", title="Top 15 Features by PSI (Higher = More Drift)")
                sections.append(("Data Drift – PSI Plot", _plotly_to_div(fig)))
        else:
            sections.append(("Drift (PSI)", "<pre class='code'>Skipped. Provide drift_ref and drift_cur (e.g., train vs test / OOT).</pre>"))

        sections.append(("Concept Drift",
                         "<pre class='code'>Concept drift requires monitoring P(Y|X) over time. This lite version provides PSI/data drift + integrity checks.\nIf you integrate DeepChecks later, you can add concept drift checks and train-vs-validation comparisons here.</pre>"))

        html = _wrap_html("Report #5 – Drift & Data Quality", sections)
        return _save_html(out_dir / "report_5_drift_quality.html", html)


# ============================================================
# USAGE EXAMPLES
# ============================================================

# --- Binary Classification ---
# cfg = FidxReportConfig(problem_type="classification", bins=10)  # bins=100 for percentiles
# fx = FidxReports4(cfg).fit(model, X_train, y_train, target_name="target")
# saved = fx.generate(
#     output_dir="outputs",
#     run_name="clf_run",
#     X_test=X_test, y_test=y_test,
#     X_train=X_train, y_train=y_train,
#     X_train_raw=X_train, X_test_raw=X_test,   # optional: add feature means to bin tables
#     which_reports="all",                      # or [1,3,5]
#     report_on="both",                         # include train + test bin tables inside Report #1
#     shap_local_index=0,
#     pdp_features=[0, 1],                      # optional
#     drift_ref=X_train, drift_cur=X_test,      # optional PSI
#     show_progress=True,
# )
# saved

# --- Regression ---
# cfg = FidxReportConfig(problem_type="regression", bins=100)
# fx = FidxReports4(cfg).fit(model, X_train, y_train, target_name="target")
# saved = fx.generate(
#     output_dir="outputs",
#     run_name="reg_run",
#     X_test=X_test, y_test=y_test,
#     X_train=X_train, y_train=y_train,
#     X_train_raw=X_train, X_test_raw=X_test,
#     which_reports="all",
#     report_on="both",
#     shap_local_index=0,
#     pdp_features=[0, 1],
#     drift_ref=X_train, drift_cur=X_test,
# )
# saved

In [35]:
!pip install xgboost==3.0.5

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 6.2 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: xgboost
    Found existing installation: xgboost 3.2.0
    Uninstalling xgboost-3.2.0:
      Successfully uninstalled xgboost-3.2.0

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [36]:
!pip list

Package                   Version
------------------------- -----------
appnope                   0.1.4
asttokens                 3.0.1
attrs                     25.4.0
certifi                   2026.1.4
charset-normalizer        3.4.4
cloudpickle               3.1.2
comm                      0.2.3
contourpy                 1.3.2
cycler                    0.12.1
debugpy                   1.8.20
decorator                 5.2.1
dice_ml                   0.12
eli5                      0.16.0
exceptiongroup            1.3.1
executing                 2.2.1
fastjsonschema            2.21.2
fonttools                 4.61.1
graphviz                  0.21
idna                      3.11
ImageIO                   2.37.2
ipykernel                 7.2.0
ipython                   8.38.0
jedi                      0.19.2
Jinja2                    3.1.6
joblib                    1.5.3
jsonschema                4.26.0
jsonschema-specifications 2025.9.1
jupyter_client            8.8.0
jupyter_core       

In [ ]:
!pip 

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

# 1. Load a sample classification dataset
# X contains the features, y contains the labels
X, y = load_breast_cancer(return_X_y=True)


# 2. Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Initialize the XGBoost classifier
# XGBoost provides a scikit-learn compatible estimator interface
model = XGBClassifier(objective='binary:logistic', use_label_encoder=False, eval_metric='logloss')

# 4. Fit the model to the training data
model.fit(X_train, y_train)

# 5. Make predictions on the test data
y_pred = model.predict(X_test)

# 6. Evaluate the model (optional)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy*100:.2f}%")

Accuracy: 95.61%


/Users/ajayparasa/.pyenv/versions/ModelXAI_p310/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [09:15:55] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [3]:
from modelxlite import modelxReportConfig, modelxLiteProject

# Example usage (binary classification):
# model = ... trained sklearn classifier with predict_proba
# X_train, y_train, X_test, y_test should be ready

cfg = modelxReportConfig(problem_type="classification", bins=10)

fx = modelxLiteProject(cfg).fit(
    model=model,
    X_train=X_train,
    y_train=y_train,
    target_name="target"
)

saved = fx.generate(
    output_dir="outputs",
    run_name="demo_run",
    X_test=X_test,
    y_test=y_test,
    X_train=X_train,
    y_train=y_train,
    X_train_raw=X_train,   # optional for feature means in bin tables
    X_test_raw=X_test,
    which_reports="all",    # or [1,2,3,4]
    report_on="both",       # "test" | "train" | "both"
    shap_local_index=0,
    pdp_features=[0, 1],
    drift_ref=X_train,
    drift_cur=X_test,
)

print(saved)


/Users/ajayparasa/.pyenv/versions/ModelXAI_p310/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning:

'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.



interval columns not set, guessing: ['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11', 'f12', 'f13', 'f14', 'f15', 'f16', 'f17', 'f18', 'f19', 'f20', 'f21', 'f22', 'f23', 'f24', 'f25', 'f26', 'f27', 'f28', 'f29', 'target']


/Users/ajayparasa/.pyenv/versions/ModelXAI_p310/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning:

'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.

/Users/ajayparasa/.pyenv/versions/ModelXAI_p310/lib/python3.10/site-packages/sklearn/inspection/_plot/partial_dependence.py:995: UserWarning:

Attempting to set identical low and high ylims makes transformation singular; automatically expanding.

/Users/ajayparasa/.pyenv/versions/ModelXAI_p310/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

divide by zero encountered in matmul

/Users/ajayparasa/.pyenv/versions/ModelXAI_p310/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

overflow encountered in matmul

/Users/ajayparasa/.pyenv/versions/ModelXAI_p310/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

invalid value encountered in matmul

/Users/ajayparasa/.pyenv/versio

{'01_model_performance': PosixPath('outputs/demo_run/01_model_performance.html'), '02_interpretability': PosixPath('outputs/demo_run/02_interpretability.html'), '03_counterfactuals': PosixPath('outputs/demo_run/03_counterfactuals.html'), '04_drift_quality': PosixPath('outputs/demo_run/04_drift_quality.html')}


In [21]:
import pandas as pd
import numpy as np

# If you have the training dataframe columns available:
feature_names = list(X_train.columns) if isinstance(X_train, pd.DataFrame) else None

def to_df(X, feature_names=None):
    if isinstance(X, pd.DataFrame):
        return X
    X = np.asarray(X)
    if feature_names is None:
        feature_names = [f"f{i}" for i in range(X.shape[1])]
    return pd.DataFrame(X, columns=feature_names)

X_train_df = to_df(X_train, feature_names=feature_names)
X_test_df  = to_df(X_test,  feature_names=feature_names or X_train_df.columns.tolist())

In [26]:
X_train_df

,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,...,f20,f21,f22,f23,f24,f25,f26,f27,f28,f29
0,10.32,16.35,65.31,324.9,0.09434,0.04994,0.01012,0.005495,0.1885,0.06201,...,11.25,21.77,71.12,384.9,0.1285,0.08842,0.04384,0.02381,0.2681,0.07399
1,20.18,19.54,133.80,1250.0,0.11330,0.14890,0.21330,0.125900,0.1724,0.06053,...,22.03,25.07,146.00,1479.0,0.1665,0.29420,0.53080,0.21730,0.3032,0.08075
2,10.66,15.15,67.49,349.6,0.08792,0.04302,0.00000,0.000000,0.1928,0.05975,...,11.54,19.20,73.20,408.3,0.1076,0.06791,0.00000,0.00000,0.2710,0.06164
3,13.56,13.90,88.59,561.3,0.10510,0.11920,0.07860,0.044510,0.1962,0.06303,...,14.98,17.13,101.10,686.6,0.1376,0.26980,0.25770,0.09090,0.3065,0.08177
4,11.37,18.89,72.17,396.0,0.08713,0.05008,0.02399,0.021730,0.2013,0.05955,...,12.36,26.14,79.29,459.3,0.1118,0.09708,0.07529,0.06203,0.3267,0.06994
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
450,15.28,22.41,98.92,710.6,0.09057,0.10520,0.05375,0.032630,0.1727,0.06317,...,17.80,28.03,113.80,973.1,0.1301,0.32990,0.36300,0.12260,0.3175,0.09772
451,19.53,18.90,129.50,1217.0,0.11500,0.16420,0.21970,0.106200,0.1792,0.06552,...,25.93,26.24,171.10,2053.0,0.1495,0.41160,0.61210,0.19800,0.2968,0.09929
452,15.46,23.95,103.80,731.3,0.11830,0.18700,0.20300,0.085200,0.1807,0.07083,...,17.11,36.33,117.70,909.4,0.1732,0.49670,0.59110,0.21630,0.3013,0.10670
453,17.05,19.08,113.40,895.0,0.11410,0.15720,0.19100,0.109000,0.2131,0.06325,...,19.59,24.89,133.50,1189.0,0.1703,0.39340,0.50180,0.25430,0.3109,0.09061


In [22]:
bad_obj = [c for c in X_test_df.columns if X_test_df[c].dtype == "object"]
print("Object dtype columns:", bad_obj[:30], " ... count:", len(bad_obj))

Object dtype columns: []  ... count: 0


In [32]:
X_train_df.shape

(455, 30)

In [33]:
X_train_df.select_dtypes(exclude='number').columns

Index([], dtype='object')

In [24]:
model = XGBClassifier(objective='binary:logistic', use_label_encoder=False, eval_metric='logloss')

# 4. Fit the model to the training data
model.fit(X_train_df, y_train)

/Users/ajayparasa/.pyenv/versions/ModelXAI_p310/lib/python3.10/site-packages/xgboost/training.py:200: UserWarning:

[08:11:19] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.




,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [28]:
shap.TreeExplainer(model)

ValueError: could not convert string to float: '[6.2637365E-1]'

In [25]:
import shap

explainer = shap.TreeExplainer(model)
Xs = X_test_df.sample(min(len(X_test_df), 1500), random_state=42)

sv = explainer.shap_values(Xs)
# For binary, sv can be list [class0, class1]
if isinstance(sv, list) and len(sv) > 1:
    sv_plot = sv[1]
else:
    sv_plot = sv

shap.summary_plot(sv_plot, Xs, show=False)

ValueError: could not convert string to float: '[6.2637365E-1]'

In [6]:
X_train_df = pd.DataFrame(X_train, columns=[f"f{i}" for i in range(X_train.shape[1])])

train_df = pd.concat(
    [X_train_df, pd.Series(y_train, name="target")],
    axis=1
)

X_test_df = pd.DataFrame(X_test, columns=X_train_df.columns)

In [5]:
from modelxlite import modelxReportConfig, modelxLiteProject

# Example usage (binary classification):
# model = ... trained sklearn classifier with predict_proba
# X_train, y_train, X_test, y_test should be ready

cfg = modelxReportConfig(problem_type="classification", bins=10)

fx = modelxLiteProject(cfg).fit(
    model=model,
    X_train=X_train,
    y_train=y_train,
    target_name="target"
)

saved = fx.generate(
    output_dir="outputs",
    run_name="demo_run",
    X_test=X_test,
    y_test=y_test,
    X_train=X_train,
    y_train=y_train,
    X_train_raw=X_train,   # optional for feature means in bin tables
    X_test_raw=X_test,
    which_reports="all",    # or [1,2,3,4]
    report_on="both",       # "test" | "train" | "both"
    shap_local_index=0,
    pdp_features=[0, 1],
    drift_ref=X_train,
    drift_cur=X_test,
)

print(saved)


/Users/ajayparasa/.pyenv/versions/ModelXAI_p310/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning:

'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.



interval columns not set, guessing: ['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11', 'f12', 'f13', 'f14', 'f15', 'f16', 'f17', 'f18', 'f19', 'f20', 'f21', 'f22', 'f23', 'f24', 'f25', 'f26', 'f27', 'f28', 'f29', 'target']


/Users/ajayparasa/.pyenv/versions/ModelXAI_p310/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning:

'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.

/Users/ajayparasa/.pyenv/versions/ModelXAI_p310/lib/python3.10/site-packages/sklearn/inspection/_plot/partial_dependence.py:995: UserWarning:

Attempting to set identical low and high ylims makes transformation singular; automatically expanding.

/Users/ajayparasa/.pyenv/versions/ModelXAI_p310/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

divide by zero encountered in matmul

/Users/ajayparasa/.pyenv/versions/ModelXAI_p310/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

overflow encountered in matmul

/Users/ajayparasa/.pyenv/versions/ModelXAI_p310/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

invalid value encountered in matmul

/Users/ajayparasa/.pyenv/versio

{'01_model_performance': PosixPath('outputs/demo_run/01_model_performance.html'), '02_interpretability': PosixPath('outputs/demo_run/02_interpretability.html'), '03_counterfactuals': PosixPath('outputs/demo_run/03_counterfactuals.html'), '04_drift_quality': PosixPath('outputs/demo_run/04_drift_quality.html')}


In [8]:
cfg = FidxReportConfig(problem_type="classification", bins=10)  # bins=100 for percentiles
fx = FidxReports4(cfg).fit(model, X_train, y_train, target_name="target")
saved = fx.generate(
    output_dir="outputs",
    run_name="clf_run",
    X_test=X_test, y_test=y_test,
    X_train=X_train, y_train=y_train,
    X_train_raw=X_train, X_test_raw=X_test,   # optional: add feature means to bin tables
    which_reports=[2] #"all",                      # or [1,3,5]
    report_on="both",                         # include train + test bin tables inside Report #1
    shap_local_index=0,
    pdp_features=[0, 1],                      # optional
    drift_ref=X_train, drift_cur=X_test,      # optional PSI
    show_progress=True,
)
saved


Generating reports:   0%|          | 0/4 [00:00<?, ?it/s, Report #1 – Model Performance Evaluation]/Users/ajayparasa/.pyenv/versions/ModelXAI_p310/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning:

'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.

/Users/ajayparasa/.pyenv/versions/ModelXAI_p310/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning:

'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.

Generating reports:  25%|██▌       | 1/4 [00:00<00:01,  1.96it/s, Report #3 – Interpretability (Global + Local)]/var/folders/3w/v9qkhfyj0n39yf1kt4hsx1yw0000gn/T/ipykernel_88217/3701467882.py:789: FutureWarning:

The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this war

{1: PosixPath('outputs/clf_run/report_1_model_performance.html'),
 3: PosixPath('outputs/clf_run/report_3_interpretability.html'),
 4: PosixPath('outputs/clf_run/report_4_counterfactuals.html'),
 5: PosixPath('outputs/clf_run/report_5_drift_quality.html')}

In [7]:
cfg = FidxReportConfig(problem_type="classification", bins=10)  # bins=100 for percentiles
fx = FidxFiveReports(cfg).fit(model, X_train, y_train, target_name="target")

saved = fx.generate(
    output_dir="outputs",
    run_name="my_classifier_run",
    X_test=X_test, y_test=y_test,
    X_train=X_train, y_train=y_train,
    X_train_raw=X_train, X_test_raw=X_test,     # optional: include feature means in bins table
    which_reports="all",                        # or [1,2,3] etc
    report_on="both",                           # include train + test bins in report #2
    shap_local_index=0,
    pdp_features=[0, 1],                        # optional
    drift_ref=X_train, drift_cur=X_test,        # for report #5 PSI drift
    show_progress=True,
)

saved


Generating 5 reports:   0%|          | 0/5 [00:00<?, ?it/s, Report #2 – Deep Evaluation / Diagnostics]/Users/ajayparasa/.pyenv/versions/ModelXAI_p310/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning:

'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.

/Users/ajayparasa/.pyenv/versions/ModelXAI_p310/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning:

'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.

Generating 5 reports:  20%|██        | 1/5 [00:00<00:00, 12.00it/s, Report #2 – Deep Evaluation / Diagnostics]


AttributeError: 'FidxReportConfig' object has no attribute 'threshold_grid'